# SAR Data Preprocessing for Oil Spill Detection

This notebook implements comprehensive preprocessing pipeline for SAR imagery including:
- SAR-specific noise reduction
- Intensity normalization
- Data augmentation strategies
- Dataset preparation for training

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. SAR Image Loading and Basic Processing

In [ ]:
class SARImageProcessor:
    """SAR Image preprocessing utilities"""
    
    def __init__(self, target_size=(512, 512)):
        self.target_size = target_size
        
    def load_sar_image(self, image_path):
        """Load SAR image with proper handling of different formats"""
        try:
            # Try rasterio first for GeoTIFF and other geo formats
            with rasterio.open(image_path) as src:
                image = src.read(1)  # Read first band
                return image.astype(np.float32)
        except:
            # Fallback to OpenCV for standard formats
            image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            return image.astype(np.float32) if image is not None else None
    
    def speckle_reduction(self, image, method='lee', window_size=5):
        """Apply speckle reduction filters specific to SAR imagery"""
        if method == 'lee':
            return self._lee_filter(image, window_size)
        elif method == 'frost':
            return self._frost_filter(image, window_size)
        elif method == 'median':
            return cv2.medianBlur(image.astype(np.uint8), window_size)
        else:
            return image
    
    def _lee_filter(self, image, window_size=5):
        """Implement Lee filter for speckle reduction"""
        kernel = np.ones((window_size, window_size)) / (window_size * window_size)
        
        # Local mean
        local_mean = cv2.filter2D(image, -1, kernel)
        
        # Local variance
        local_sqr_mean = cv2.filter2D(image**2, -1, kernel)
        local_variance = local_sqr_mean - local_mean**2
        
        # Lee filter coefficient
        noise_variance = np.var(image)  # Estimate noise variance
        k = local_variance / (local_variance + noise_variance + 1e-8)
        
        # Apply filter
        filtered = local_mean + k * (image - local_mean)
        return filtered
    
    def _frost_filter(self, image, window_size=5):
        """Implement Frost filter for speckle reduction"""
        # Simplified Frost filter implementation
        kernel = np.ones((window_size, window_size)) / (window_size * window_size)
        return cv2.filter2D(image, -1, kernel)
    
    def normalize_intensity(self, image, method='percentile'):
        """Normalize SAR image intensities"""
        if method == 'percentile':
            # Use percentile normalization to handle outliers
            p2, p98 = np.percentile(image, (2, 98))
            image_norm = np.clip((image - p2) / (p98 - p2 + 1e-8), 0, 1)
        elif method == 'log':
            # Log transformation for SAR data
            image_norm = np.log10(image + 1)
            image_norm = (image_norm - image_norm.min()) / (image_norm.max() - image_norm.min() + 1e-8)
        else:
            # Standard min-max normalization
            image_norm = (image - image.min()) / (image.max() - image.min() + 1e-8)
            
        return image_norm
    
    def enhance_contrast(self, image, method='clahe'):
        """Enhance contrast for better oil spill visibility"""
        if method == 'clahe':
            # Convert to uint8 for CLAHE
            image_uint8 = (image * 255).astype(np.uint8)
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            enhanced = clahe.apply(image_uint8)
            return enhanced.astype(np.float32) / 255.0
        elif method == 'histogram_eq':
            image_uint8 = (image * 255).astype(np.uint8)
            enhanced = cv2.equalizeHist(image_uint8)
            return enhanced.astype(np.float32) / 255.0
        else:
            return image
    
    def process_image(self, image_path, apply_speckle_reduction=True, 
                     apply_contrast_enhancement=True):
        """Complete preprocessing pipeline"""
        # Load image
        image = self.load_sar_image(image_path)
        if image is None:
            return None
        
        # Resize to target size
        image = cv2.resize(image, self.target_size, interpolation=cv2.INTER_CUBIC)
        
        # Apply speckle reduction
        if apply_speckle_reduction:
            image = self.speckle_reduction(image, method='lee')
        
        # Normalize intensity
        image = self.normalize_intensity(image, method='percentile')
        
        # Enhance contrast
        if apply_contrast_enhancement:
            image = self.enhance_contrast(image, method='clahe')
        
        return image

## 2. Data Augmentation for SAR Images

In [ ]:
class SARDataAugmentation:
    """SAR-specific data augmentation techniques"""
    
    def __init__(self, image_size=512):
        self.image_size = image_size
        
    def get_training_transforms(self):
        """Training augmentation pipeline"""
        return A.Compose([
            A.Resize(self.image_size, self.image_size),
            
            # Geometric transformations
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Rotate(limit=15, p=0.5),
            
            # SAR-specific transformations
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),  # Simulate SAR speckle
            
            # Elastic transformations for oil spill shape variation
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
            A.GridDistortion(p=0.3),
            
            # Intensity variations
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
            
            # Cutout for robustness
            A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.2),
            
            A.Normalize(mean=[0.485], std=[0.229]),  # ImageNet normalization adapted for grayscale
            ToTensorV2(),
        ])
    
    def get_validation_transforms(self):
        """Validation/test augmentation pipeline"""
        return A.Compose([
            A.Resize(self.image_size, self.image_size),
            A.Normalize(mean=[0.485], std=[0.229]),
            ToTensorV2(),
        ])
    
    def get_segmentation_transforms(self):
        """Augmentation for segmentation tasks (image + mask)"""
        return A.Compose([
            A.Resize(self.image_size, self.image_size),
            
            # Geometric transformations (applied to both image and mask)
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Rotate(limit=15, p=0.5),
            
            # Elastic transformations
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
            A.GridDistortion(p=0.3),
            
            # Image-only transformations
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            
            A.Normalize(mean=[0.485], std=[0.229]),
            ToTensorV2(),
        ])

## 3. Dataset Organization and Preparation

In [ ]:
import os
import json
from torch.utils.data import Dataset, DataLoader
import torch
from PIL import Image

class OilSpillDataset(Dataset):
    """Dataset class for oil spill detection and segmentation"""
    
    def __init__(self, data_dir, mode='detection', split='train', transforms=None):
        """
        Args:
            data_dir: Path to dataset directory
            mode: 'detection' or 'segmentation'
            split: 'train', 'val', or 'test'
            transforms: Albumentations transforms
        """
        self.data_dir = Path(data_dir)
        self.mode = mode
        self.split = split
        self.transforms = transforms
        self.processor = SARImageProcessor()
        
        # Load metadata
        self.metadata = self._load_metadata()
        
    def _load_metadata(self):
        """Load dataset metadata"""
        metadata_file = self.data_dir / f"{self.split}_metadata.json"
        if metadata_file.exists():
            with open(metadata_file, 'r') as f:
                return json.load(f)
        else:
            # Create sample metadata if not exists
            return self._create_sample_metadata()
    
    def _create_sample_metadata(self):
        """Create sample metadata for demonstration"""
        # This would be replaced with actual dataset loading logic
        sample_data = []
        image_dir = self.data_dir / "images" / self.split
        
        if image_dir.exists():
            for img_path in image_dir.glob("*.png"):
                sample_data.append({
                    'image_path': str(img_path),
                    'has_oil_spill': 1 if 'spill' in img_path.name else 0,
                    'mask_path': str(self.data_dir / "masks" / self.split / img_path.name) if self.mode == 'segmentation' else None
                })
        
        return sample_data
    
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        item = self.metadata[idx]
        
        # Load and process image
        image = self.processor.process_image(item['image_path'])
        if image is None:
            # Return a dummy image if loading fails
            image = np.zeros((512, 512), dtype=np.float32)
        
        # Convert to 3-channel for compatibility with pre-trained models
        if len(image.shape) == 2:
            image = np.stack([image, image, image], axis=-1)
        
        if self.mode == 'detection':
            label = item['has_oil_spill']
            
            if self.transforms:
                augmented = self.transforms(image=image)
                image = augmented['image']
            
            return image, torch.tensor(label, dtype=torch.long)
        
        elif self.mode == 'segmentation':
            # Load mask
            mask_path = item.get('mask_path')
            if mask_path and os.path.exists(mask_path):
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                mask = cv2.resize(mask, (512, 512))
                mask = (mask > 127).astype(np.float32)  # Binary mask
            else:
                # Create dummy mask
                mask = np.zeros((512, 512), dtype=np.float32)
            
            if self.transforms:
                augmented = self.transforms(image=image, mask=mask)
                image = augmented['image']
                mask = augmented['mask']
            
            return image, torch.tensor(mask, dtype=torch.float32)

def create_data_loaders(data_dir, batch_size=16, num_workers=4):
    """Create data loaders for training and validation"""
    augmentation = SARDataAugmentation()
    
    # Detection datasets
    train_dataset_det = OilSpillDataset(
        data_dir, mode='detection', split='train',
        transforms=augmentation.get_training_transforms()
    )
    
    val_dataset_det = OilSpillDataset(
        data_dir, mode='detection', split='val',
        transforms=augmentation.get_validation_transforms()
    )
    
    # Segmentation datasets
    train_dataset_seg = OilSpillDataset(
        data_dir, mode='segmentation', split='train',
        transforms=augmentation.get_segmentation_transforms()
    )
    
    val_dataset_seg = OilSpillDataset(
        data_dir, mode='segmentation', split='val',
        transforms=augmentation.get_validation_transforms()
    )
    
    # Create data loaders
    loaders = {
        'detection': {
            'train': DataLoader(train_dataset_det, batch_size=batch_size, 
                              shuffle=True, num_workers=num_workers, pin_memory=True),
            'val': DataLoader(val_dataset_det, batch_size=batch_size, 
                            shuffle=False, num_workers=num_workers, pin_memory=True)
        },
        'segmentation': {
            'train': DataLoader(train_dataset_seg, batch_size=batch_size, 
                              shuffle=True, num_workers=num_workers, pin_memory=True),
            'val': DataLoader(val_dataset_seg, batch_size=batch_size, 
                            shuffle=False, num_workers=num_workers, pin_memory=True)
        }
    }
    
    return loaders

## 4. Visualization and Quality Check

In [ ]:
def visualize_preprocessing_steps(image_path, processor):
    """Visualize different preprocessing steps"""
    # Load original image
    original = processor.load_sar_image(image_path)
    if original is None:
        print(f"Could not load image: {image_path}")
        return
    
    # Resize
    resized = cv2.resize(original, (512, 512))
    
    # Apply different preprocessing steps
    speckle_reduced = processor.speckle_reduction(resized, method='lee')
    normalized = processor.normalize_intensity(speckle_reduced, method='percentile')
    enhanced = processor.enhance_contrast(normalized, method='clahe')
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    images = [resized, speckle_reduced, normalized, enhanced]
    titles = ['Original (Resized)', 'Speckle Reduced', 'Normalized', 'Enhanced']
    
    for i, (img, title) in enumerate(zip(images, titles)):
        row, col = i // 3, i % 3
        if row < 2 and col < 3:
            axes[row, col].imshow(img, cmap='gray')
            axes[row, col].set_title(title)
            axes[row, col].axis('off')
    
    # Hide unused subplots
    for i in range(len(images), 6):
        row, col = i // 3, i % 3
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

def visualize_augmentations(dataset, num_samples=4):
    """Visualize augmented samples"""
    fig, axes = plt.subplots(2, num_samples, figsize=(16, 8))
    
    for i in range(num_samples):
        # Get sample
        image, label = dataset[i]
        
        # Convert tensor back to numpy for visualization
        if isinstance(image, torch.Tensor):
            image_np = image.permute(1, 2, 0).numpy()
            # Denormalize
            image_np = image_np * 0.229 + 0.485
            image_np = np.clip(image_np, 0, 1)
            
            # Use only first channel for grayscale display
            image_np = image_np[:, :, 0]
        else:
            image_np = image
        
        axes[0, i].imshow(image_np, cmap='gray')
        axes[0, i].set_title(f'Sample {i+1} (Label: {label})')
        axes[0, i].axis('off')
        
        # Show histogram
        axes[1, i].hist(image_np.flatten(), bins=50, alpha=0.7)
        axes[1, i].set_title(f'Intensity Distribution')
        axes[1, i].set_xlabel('Intensity')
        axes[1, i].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

# Test preprocessing pipeline
print("Setting up preprocessing pipeline...")
processor = SARImageProcessor(target_size=(512, 512))
augmentation = SARDataAugmentation(image_size=512)

print("\nPreprocessing pipeline ready!")
print("Available methods:")
print("- Speckle reduction (Lee, Frost, Median filters)")
print("- Intensity normalization (Percentile, Log, Min-Max)")
print("- Contrast enhancement (CLAHE, Histogram Equalization)")
print("- SAR-specific data augmentation")

## 5. Configuration and Setup

In [ ]:
# Configuration
CONFIG = {
    'data_dir': '../data',  # TIFF images directory
    'image_size': 512,
    'batch_size': 16,
    'num_workers': 4,
    'preprocessing': {
        'apply_speckle_reduction': True,
        'speckle_method': 'lee',
        'normalization_method': 'percentile',
        'apply_contrast_enhancement': True,
        'contrast_method': 'clahe'
    },
    'augmentation': {
        'horizontal_flip': 0.5,
        'vertical_flip': 0.5,
        'rotation_limit': 15,
        'brightness_contrast': 0.5,
        'gaussian_noise': 0.3,
        'elastic_transform': 0.3
    }
}

# Save configuration
import json
config_path = 'preprocessing_config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Configuration saved to {config_path}")
print("\nNext steps:")
print("1. Prepare your SAR dataset in the specified directory structure")
print("2. Run the detection model notebook (02_Detection_Model.ipynb)")
print("3. Run the segmentation model notebook (03_Segmentation_Model.ipynb)")

# Create data directory structure
os.makedirs('../data/images/train', exist_ok=True)
os.makedirs('../data/images/val', exist_ok=True)
os.makedirs('../data/images/test', exist_ok=True)
os.makedirs('../data/masks/train', exist_ok=True)
os.makedirs('../data/masks/val', exist_ok=True)
os.makedirs('../data/masks/test', exist_ok=True)

print("\nData directory structure created!")